# Avance 5. Modelo final

## Equipo 41

**Proyecto:** Detección de ocultamiento de mercancía en video para retail

**Integrantes:**
- Jesús Alberto Jiménez Ramos
- Nélida Silva Cabrera
- José Luis Tinajero Guerrero

**Empresa:** BLOCK NETWORKS S.A. DE C.V. (BNext)

**Asesor:** Dr. Guillermo Mota Medina

**Programa:** Maestría en Inteligencia Artificial Aplicada  
**Tecnológico de Monterrey**

**Fecha:** Octubre de 2026

## 1. Objetivo

El objetivo de este avance es evaluar estrategias de ensamble para determinar si la combinación de modelos o representaciones desarrollados en las etapas anteriores permite mejorar el desempeño alcanzado por el mejor modelo individual del proyecto.

El Avance 4 permitió comparar diferentes arquitecturas para reconocimiento de acciones en video y seleccionar **R(2+1)D-18/H5** como el modelo individual de referencia con base principalmente en Average Precision. En este avance, dicho modelo se utilizará como punto de comparación para evaluar si estrategias de ensamble homogéneas y heterogéneas pueden aprovechar información complementaria y reducir algunas de las limitaciones observadas en los modelos individuales.

La evaluación considerará tanto el desempeño predictivo como el comportamiento de falsos positivos y falsos negativos, los tiempos de entrenamiento e inferencia y la complejidad adicional introducida por cada alternativa. Estos criterios permitirán seleccionar un modelo final considerando las características del problema de detección de ocultamiento de mercancía y el uso previsto de las predicciones como apoyo para generar alertas sujetas a revisión humana.

La selección final se realizará mediante un protocolo común de evaluación, conservando las particiones experimentales definidas previamente y evitando utilizar el conjunto de prueba para ajustar hiperparámetros, pesos de combinación o umbrales de decisión.

## 2. Relación entre la rúbrica y el diseño experimental

La rúbrica del Avance 5 establece como elementos principales la construcción de al menos cuatro modelos de ensamble, la inclusión de estrategias homogéneas y heterogéneas, la comparación con el mejor modelo individual de la fase anterior y la selección de un modelo final sustentada en métricas de desempeño y costo computacional.

Debido a que este proyecto trabaja con clips de video y representaciones espacio-temporales aprendidas por redes neuronales, algunos criterios requieren una adaptación al dominio. La adaptación conserva el propósito de la rúbrica, pero evita tratar directamente los frames de video como si fueran variables tabulares independientes.

| Criterio de la rúbrica | Aplicación en este proyecto |
|---|---|
| Al menos cuatro ensambles | Se evaluarán Random Forest, AdaBoost, soft voting ponderado y blending. |
| Estrategias homogéneas | Random Forest y AdaBoost se aplicarán sobre embeddings obtenidos de R(2+1)D-18/H5. Los estimadores que forman cada ensamble pertenecen a una misma familia de algoritmos. |
| Estrategias heterogéneas | Soft voting y blending combinarán las probabilidades producidas por diferentes arquitecturas RGB. |
| Uso de los mejores modelos individuales previos | R(2+1)D-18/H5 y MC3-18/H5, seleccionados durante el Avance 4, participarán en los ensambles heterogéneos. MViT-V2-S se incorporará para aportar diversidad arquitectónica. |
| Optimización de hiperparámetros | Cada estrategia será evaluada mediante un espacio de búsqueda acotado y utilizando únicamente los datos permitidos para entrenamiento y validación. |
| Comparación con el mejor modelo individual | R(2+1)D-18/H5 se conservará como referencia para determinar si el aumento de complejidad de los ensambles produce una mejora justificable. |
| Métrica principal | Average Precision continuará como criterio principal para conservar comparabilidad con los avances anteriores. |
| Métricas adicionales | Se considerarán PR-AUC, ROC-AUC, EER, precision, recall, specificity y F1-score, además de los conteos TP, TN, FP y FN. |
| Tiempos de entrenamiento | Se registrará el tiempo de entrenamiento de los ensambles y, cuando corresponda, el costo requerido para generar las representaciones o predicciones de los modelos base. |
| Selección del modelo final | La selección considerará desempeño, tipos de error, costo computacional y alineación con el uso previsto por BNext. |
| Gráficas | Se generarán al menos cuatro visualizaciones relevantes del modelo seleccionado, acompañadas por una interpretación de sus implicaciones. |

STG-NF no se incorporará directamente a los ensambles de este avance. Aunque fue evaluado en el Avance 4, utiliza un conjunto de datos diferente, ventanas de pose como unidad de análisis y un enfoque de detección de anomalías. Por ello, sus scores no están alineados directamente con las probabilidades por clip producidas por los modelos RGB.

## 3. Punto de partida del Avance 5

El Avance 5 continúa a partir del protocolo experimental y de los modelos obtenidos en el Avance 4. Para mantener la comparabilidad entre ambas etapas, se conservarán sin modificaciones el dataset v2, sus particiones experimentales y los checkpoints seleccionados de los modelos RGB.

### 3.1 Dataset y particiones

El dataset v2 mantiene los 11 casos identificados previamente como ambiguos fuera de la evaluación binaria. Las observaciones elegibles permanecen distribuidas de la siguiente manera:

| Partición | Total | Sin ocultamiento | Ocultamiento |
|---|---:|---:|---:|
| Train | 702 | 570 | 132 |
| Validation | 136 | 106 | 30 |
| Test | 136 | 119 | 17 |

Los grupos experimentales permanecen separados entre `train`, `validation` y `test`. Esta condición se conservará durante todos los experimentos del Avance 5 para evitar que clips relacionados aparezcan en más de una partición.

### 3.2 Modelo individual de referencia

El ajuste de hiperparámetros del Avance 4 identificó **R(2+1)D-18/H5** como el modelo individual con mejor desempeño en `validation`.

Los dos modelos finalistas optimizados obtuvieron:

| Modelo | AP validation | PR-AUC | ROC-AUC | EER |
|---|---:|---:|---:|---:|
| R(2+1)D-18/H5 | 0.7554 | 0.7684 | 0.8668 | 0.2299 |
| MC3-18/H5 | 0.7134 | 0.7112 | 0.8302 | 0.2701 |

R(2+1)D-18/H5 se conservará como **modelo individual de referencia** para determinar si las estrategias de ensamble aportan una mejora suficiente para justificar su complejidad adicional.

### 3.3 Referencia de desempeño en test

Después de congelar su configuración y seleccionar el threshold utilizando `validation`, R(2+1)D-18/H5 fue evaluado en `test v2`.

El modelo obtuvo un Average Precision de **0.7551**. Con el threshold experimental de **0.9971**, alcanzó una precision de **0.7857**, recall de **0.6471** y specificity de **0.9748**, con **3 falsos positivos y 6 falsos negativos**.

Estos resultados se conservarán como referencia histórica del mejor modelo individual.

Debe considerarse que `test v2` ya fue utilizado para la evaluación final del Avance 4. Por esta razón, durante el Avance 5 no se utilizará información de `test` para seleccionar hiperparámetros, pesos de combinación, meta-modelos o thresholds. Las decisiones de diseño deberán realizarse utilizando únicamente la información permitida de entrenamiento y validación.

### 3.4 Modelos base disponibles

Los ensambles del Avance 5 reutilizarán checkpoints generados durante el Avance 4. Los principales modelos considerados como punto de partida son:

- **R(2+1)D-18/H5**, como mejor modelo individual.
- **MC3-18/H5**, como segundo modelo optimizado.
- **MViT-V2-S**, como arquitectura RGB adicional con un mecanismo de representación diferente.

Los pesos de estos modelos se mantendrán congelados cuando se utilicen para generar embeddings o probabilidades destinadas a los ensambles. De esta forma, las diferencias observadas en el Avance 5 podrán atribuirse al procedimiento de ensamblado y no a modificaciones adicionales de los modelos base.

## 4. Definiciones y decisiones de modelado

### 4.1 Unidad de análisis

La unidad de análisis se mantiene como el **clip de video** utilizado en los avances anteriores. Cada clip corresponde a una observación independiente del clasificador y recibe una etiqueta binaria:

- `0`: sin ocultamiento.
- `1`: ocultamiento.

Esta unidad permite mantener correspondencia directa entre las observaciones utilizadas por los diferentes modelos RGB y es necesaria para combinar posteriormente sus predicciones.

### 4.2 Embeddings

Un **embedding** es una representación numérica interna aprendida por una red neuronal a partir del clip de entrada.

Para un clip $x_i$, una red de video puede representarse de forma simplificada como:

$$
z_i = f_{\theta}(x_i)
$$

donde:

- $x_i$ es el clip de video.
- $f_{\theta}$ representa el backbone entrenado.
- $z_i$ es el embedding generado antes del clasificador final.

En este avance se extraerán los embeddings de **R(2+1)D-18/H5** antes de su última capa de clasificación.

La finalidad es utilizar la información espacio-temporal aprendida por la red como entrada para modelos de ensamble clásicos, en lugar de utilizar directamente los pixeles del video.

### 4.3 Probabilidades de los modelos base

Después del embedding, el clasificador final transforma la representación interna en scores de clase y posteriormente en una probabilidad de `ocultamiento`.

De forma simplificada:

$$
x_i
\rightarrow
z_i
\rightarrow
P(y_i = 1 \mid x_i)
$$

Para cada clip, los modelos base producirán probabilidades como:

$$
p_{R(2+1)D},\quad
p_{MC3},\quad
p_{MViT}
$$

Estas probabilidades constituyen las entradas de los ensambles heterogéneos.

El embedding conserva una representación multidimensional del contenido del clip, mientras que la probabilidad constituye una salida resumida del clasificador para una clase específica.

### 4.4 Ensambles homogéneos

Los ensambles homogéneos combinan estimadores pertenecientes a una misma familia de algoritmos.

En este avance se evaluarán dos estrategias:

- **Random Forest**, basado en múltiples árboles de decisión entrenados mediante una estrategia de bagging;
- **AdaBoost**, basado en una secuencia de estimadores donde los modelos posteriores buscan corregir errores cometidos por los estimadores anteriores.

Ambos modelos recibirán como entrada los embeddings extraídos de **R(2+1)D-18/H5**.

El flujo puede representarse como:

$$
x_i
\rightarrow
z_i
\rightarrow
g(z_i)
\rightarrow
P(y_i = 1)
$$

donde:

- $x_i$ corresponde al clip de video.
- $z_i$ corresponde al embedding generado por R(2+1)D-18/H5.
- $g(\cdot)$ corresponde al modelo de ensamble.
- $P(y_i = 1)$ corresponde a la probabilidad estimada de `ocultamiento`.

Por lo tanto, Random Forest y AdaBoost no procesarán directamente los frames RGB. Primero, el backbone de video transformará cada clip en una representación numérica. Posteriormente, cada algoritmo de ensamble aprenderá una función de clasificación utilizando esa representación.

Esta adaptación permite utilizar métodos clásicos de ensamble sobre información obtenida de un problema de video sin tratar los pixeles individuales como variables tabulares independientes.

### 4.5 Ensambles heterogéneos

Los ensambles heterogéneos aprovechan modelos de diferentes arquitecturas con el propósito de utilizar información potencialmente complementaria.

En este proyecto se considerarán:

- **R(2+1)D-18/H5**.
- **MC3-18/H5**.
- **MViT-V2-S**.

R(2+1)D-18 y MC3-18 corresponden a arquitecturas convolucionales diseñadas para procesar información espacio-temporal en video, mientras que MViT-V2-S incorpora un enfoque Transformer multiescala.

La inclusión de modelos con arquitecturas distintas busca introducir diversidad en el ensamble.

Un modelo con menor desempeño individual puede seguir aportando información útil si clasifica correctamente observaciones en las que otros modelos se equivocan. Por esta razón, además del desempeño individual, posteriormente se analizará el grado de complementariedad entre las predicciones de los modelos base.

### 4.6 Soft voting ponderado

El **soft voting** combina directamente las probabilidades generadas por los modelos base.

Para los tres modelos considerados, la combinación puede expresarse como:

$$
p_{\text{ensamble}}
=
w_1 p_{R(2+1)D}
+
w_2 p_{MC3}
+
w_3 p_{MViT}
$$

donde:

- $w_1$ representa el peso asignado a R(2+1)D-18/H5.
- $w_2$ representa el peso asignado a MC3-18/H5.
- $w_3$ representa el peso asignado a MViT-V2-S.
- $p_{\text{ensamble}}$ corresponde a la probabilidad combinada de `ocultamiento`.

Los pesos representan la contribución relativa de cada modelo al resultado conjunto.

Como referencia inicial puede utilizarse un promedio uniforme:

$$
w_1 = w_2 = w_3
$$

Posteriormente podrán evaluarse combinaciones de pesos utilizando exclusivamente información permitida de validación.

Los pesos seleccionados deberán permanecer congelados antes de realizar la evaluación final sobre `test`.

### 4.7 Blending

En **blending**, las probabilidades generadas por los modelos base se convierten en las variables de entrada de un nuevo modelo denominado **meta-modelo**.

El proceso puede representarse como:

$$
[p_{R(2+1)D}, p_{MC3}, p_{MViT}]
\rightarrow
g(\cdot)
\rightarrow
p_{\text{ensamble}}
$$

donde:

- $p_{R(2+1)D}$ corresponde a la probabilidad generada por R(2+1)D-18/H5.
- $p_{MC3}$ corresponde a la probabilidad generada por MC3-18/H5.
- $p_{MViT}$ corresponde a la probabilidad generada por MViT-V2-S.
- $g(\cdot)$ representa el meta-modelo.
- $p_{\text{ensamble}}$ representa la probabilidad final generada por el ensamble.

A diferencia del soft voting, el blending no queda limitado a utilizar un promedio de probabilidades con pesos definidos explícitamente.

El meta-modelo aprende, a partir de los datos permitidos para esta etapa, cómo utilizar conjuntamente las predicciones de los modelos base.

Esto permite capturar relaciones entre las salidas de los modelos. Por ejemplo, el meta-modelo puede aprender que determinadas combinaciones de probabilidades proporcionan evidencia diferente a la que se obtendría mediante un promedio directo.

Debido a que el blending introduce un segundo nivel de aprendizaje, su diseño deberá evitar que el meta-modelo sea entrenado y evaluado sobre las mismas observaciones.

### 4.8 Exclusión de STG-NF de los ensambles directos

STG-NF no se incorporará directamente al soft voting o al blending desarrollados en este avance.

El modelo STG-NF utiliza:

- Ventanas temporales de pose.
- Coordenadas corporales como representación.
- Un enfoque de detección de anomalías.

En contraste, los modelos R(2+1)D-18, MC3-18 y MViT-V2-S utilizan:

- Clips RGB.
- Información visual espacio-temporal.
- Clasificación binaria supervisada.

Además, los experimentos con STG-NF fueron realizados utilizando conjuntos y unidades de análisis diferentes.

Para realizar soft voting o blending se requiere que las predicciones de todos los modelos correspondan a la misma observación:

$$
x_i
\rightarrow
[p_1, p_2, p_3]
$$

En el protocolo experimental actual no existe esta correspondencia directa entre las ventanas de pose utilizadas por STG-NF y los clips RGB utilizados por los demás modelos.

Por esta razón, sus scores no se combinarán directamente con las probabilidades de los modelos RGB.

Esta exclusión no implica que el enfoque de pose carezca de valor para el problema. Significa únicamente que su incorporación requeriría un diseño de fusión diferente que permita alinear ambas representaciones de manera experimentalmente válida.

### 4.9 Score, probabilidad y threshold

Los modelos generan inicialmente un score o una probabilidad continua asociada con la clase `ocultamiento`.

Posteriormente, un **threshold** transforma esta salida continua en una decisión binaria.

La regla de decisión puede expresarse como:

$$
\hat{y} =
\begin{cases}
1, & p \geq t \\
0, & p < t
\end{cases}
$$

donde:

- $p$ representa la probabilidad estimada de `ocultamiento`.
- $t$ representa el threshold seleccionado.
- $\hat{y}$ representa la clase predicha.

Por esta razón se distinguen dos decisiones:

1. evaluar qué tan bien el modelo separa u ordena los casos positivos y negativos;
2. seleccionar posteriormente un punto de operación para transformar sus scores en decisiones binarias.

Modificar el threshold no modifica los parámetros aprendidos por el modelo. Únicamente cambia la frontera utilizada para convertir sus scores en decisiones.

Por ejemplo, aumentar $t$ normalmente hace más estricta la condición necesaria para producir una predicción positiva. Esto puede disminuir el número de falsos positivos, pero también puede aumentar el número de falsos negativos.

Esta relación resulta especialmente relevante para el caso de BNext, debido a que las predicciones positivas se utilizarían como apoyo para generar alertas sujetas a revisión humana.

### 4.10 Resumen del diseño de ensambles

El diseño experimental del Avance 5 puede resumirse en dos rutas principales.

**Ruta basada en representaciones:**

$$
\text{Clip RGB}
\rightarrow
\text{R(2+1)D-18/H5}
\rightarrow
\text{Embedding}
\rightarrow
\begin{cases}
\text{Random Forest} \\
\text{AdaBoost}
\end{cases}
\rightarrow
p_{\text{ocultamiento}}
$$

**Ruta basada en predicciones:**

$$
\text{Clip RGB}
\rightarrow
\begin{cases}
\text{R(2+1)D-18/H5} \\
\text{MC3-18/H5} \\
\text{MViT-V2-S}
\end{cases}
\rightarrow
[p_1,p_2,p_3]
\rightarrow
\begin{cases}
\text{Soft Voting} \\
\text{Blending}
\end{cases}
\rightarrow
p_{\text{ocultamiento}}
$$

La primera ruta busca aprovechar la información contenida en una representación espacio-temporal aprendida por una red de video.

La segunda busca aprovechar la complementariedad entre las decisiones generadas por diferentes arquitecturas.

## 5. Criterios de calidad experimental

El diseño experimental del Avance 5 busca que las diferencias observadas entre el mejor modelo individual y los ensambles puedan atribuirse principalmente a la estrategia de combinación utilizada. Para ello se conservarán las mismas particiones, etiquetas y criterios de evaluación establecidos en el Avance 4.

### 5.1 Control de datos y particiones

Se utilizará el mismo dataset v2 congelado en el Avance 4, identificado mediante el SHA 256:

`b85d6b55e8c32420a4df5a0fb9a10e558105beee47f1d4a98ad83d63fcbda21e`

Los 63 grupos experimentales permanecen separados entre `train`, `validation` y `test`, evitando que clips relacionados con una misma secuencia aparezcan en más de una partición.

Cada conjunto conservará una función específica:

1. `Train` para ajustar los modelos que requieran aprendizaje.
2. `Validation` para seleccionar hiperparámetros, pesos de combinación, meta modelos y threshold.
3. `Test` para evaluar únicamente una configuración previamente congelada.

Aunque `test v2` ya fue observado en el Avance 4, durante este avance no se utilizará para modificar decisiones de modelado.

### 5.2 Prevención de data leakage

Los checkpoints de R(2+1)D-18/H5, MC3-18/H5 y MViT-V2-S se mantendrán congelados cuando se utilicen para generar embeddings o probabilidades.

Los embeddings conservarán `clip_id`, etiqueta, partición y grupo experimental para mantener su trazabilidad.

Los embeddings de `validation` y `test` no se utilizarán para entrenar Random Forest o AdaBoost.

En blending se evitará entrenar y evaluar el meta modelo sobre las mismas observaciones. Cualquier división interna deberá conservar también la independencia entre grupos experimentales.

### 5.3 Comparabilidad y reproducibilidad

Todos los modelos se evaluarán con las mismas observaciones elegibles, etiquetas y funciones de métricas. Average Precision continuará como métrica principal para mantener comparabilidad con los avances anteriores.

La búsqueda de hiperparámetros será acotada para reducir el riesgo de adaptar excesivamente las configuraciones al conjunto de validación.

Se registrarán los principales elementos necesarios para reproducir los experimentos:

1. versión y SHA 256 del dataset
2. semilla aleatoria
3. checkpoints utilizados
4. hiperparámetros
5. configuración seleccionada
6. tiempos de entrenamiento e inferencia
7. versiones principales de las bibliotecas
8. threshold seleccionado
9. archivos de embeddings y probabilidades

### 5.4 Costo computacional y alcance de los resultados

Una mejora en las métricas deberá analizarse junto con el costo adicional del ensamble.

Los ensambles heterogéneos requieren ejecutar varios modelos base antes de producir una predicción final. Por esta razón se considerarán tanto el tiempo adicional del ensamble como el costo conjunto de inferencia de los modelos que lo forman.

Los resultados obtenidos representan evidencia experimental dentro del conjunto de datos disponible. No constituyen por sí mismos evidencia suficiente de desempeño operativo en tiendas reales. La selección final deberá considerar las métricas, los tipos de error, el costo computacional y las limitaciones del dataset.

## 6. Métricas de evaluación

La evaluación del Avance 5 conservará las métricas utilizadas en el Avance 4 para mantener comparabilidad entre el mejor modelo individual y los modelos de ensamble.

### 6.1 Métrica principal

**Average Precision (AP)** continuará como la métrica principal para seleccionar y ordenar los modelos.

$$
AP = \sum_n (R_n - R_{n-1})P_n
$$

donde $P_n$ y $R_n$ representan precision y recall en diferentes niveles de decisión.

AP resume el comportamiento de precision y recall a través de múltiples thresholds. Esto resulta adecuado para el proyecto debido al desbalance entre `ocultamiento` y `sin_ocultamiento` y permite comparar la capacidad de discriminación antes de seleccionar un punto de operación específico.

### 6.2 Métricas complementarias independientes del threshold

**PR AUC** representa el área bajo la curva Precision Recall. Se reportará junto con AP para caracterizar el comportamiento del modelo sobre la clase positiva.

**ROC AUC** mide la capacidad de separar positivos y negativos considerando múltiples thresholds. Un valor mayor indica una mejor capacidad discriminativa.

**Equal Error Rate (EER)** representa aproximadamente el punto donde la tasa de falsos positivos y la tasa de falsos negativos son iguales. En esta métrica un valor menor representa un mejor resultado.

Estas métricas permiten comparar modelos sin depender todavía de un threshold operativo concreto.

### 6.3 Métricas dependientes del threshold

Una vez seleccionado un threshold se calcularán las siguientes métricas:

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Recall = \frac{TP}{TP + FN}
$$

$$
Specificity = \frac{TN}{TN + FP}
$$

$$
F1 =
2
\frac{Precision \cdot Recall}
{Precision + Recall}
$$

**Precision** indica qué proporción de las alertas generadas corresponde realmente a eventos positivos.

**Recall** indica qué proporción de los eventos reales de ocultamiento fue detectada.

**Specificity** indica qué proporción de los casos reales `sin_ocultamiento` fue correctamente descartada.

**F1** resume el equilibrio entre precision y recall para un threshold específico.

También se reportarán los valores absolutos de **TP, TN, FP y FN**. Estos conteos son especialmente importantes para interpretar el modelo en el contexto de BNext. Los falsos positivos se relacionan con alertas innecesarias y los falsos negativos representan eventos de ocultamiento que el sistema no detecta.

Accuracy se conservará únicamente como referencia descriptiva. Debido al desbalance de clases, no se utilizará como criterio principal para seleccionar los ensambles.

### 6.4 Métricas computacionales

La comparación incluirá también el costo computacional de cada alternativa.

Se registrarán, cuando corresponda:

1. Tiempo de entrenamiento.
2. Tiempo de generación de embeddings o probabilidades.
3. Velocidad o tiempo de inferencia.
4. Cantidad de modelos base requeridos durante inferencia.

Este análisis permitirá determinar si una posible mejora predictiva de los ensambles justifica el aumento de complejidad respecto a R(2+1)D-18/H5 utilizado de forma individual.

## 7. Preparación reproducible del entorno

Antes de iniciar los experimentos de ensamble se configura y verifica el entorno de ejecución. Esta sección establece las condiciones necesarias para reproducir los resultados y confirma que el notebook puede acceder a los recursos del proyecto sin depender de rutas personales.

### 7.1 Configuración global

Se cargan las bibliotecas principales, se fija la semilla experimental y se establece el control que impide ejecutar entrenamientos costosos de manera accidental durante `Restart Kernel + Run All`.

In [1]:
# ============================================================
# Librerías y configuración global
# ============================================================

from pathlib import Path
import os
import sys
import random

import numpy as np
import pandas as pd

import torch
import torchvision
import sklearn


SEED = 42

# Evita entrenamientos costosos durante Restart Kernel + Run All
RUN_TRAINING = False


def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


fijar_semilla(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

### 7.2 Localización del repositorio

El notebook localiza automáticamente la raíz del repositorio a partir de su estructura interna. Esto permite utilizar rutas relativas y ejecutar la libreta desde diferentes entornos sin incorporar rutas personales o absolutas.

In [2]:
# ============================================================
# Localización de la raíz del repositorio
# ============================================================

MARCADOR_REPOSITORIO = (
    Path("Datos")
    / "Preparacion_Datasets"
    / "avance4_split_ground_truth_v2.csv"
)


def find_repo_root():
    candidatos = []

    repo_env = os.environ.get("MNA_REPO_DIR")

    if repo_env:
        candidatos.append(Path(repo_env))

    candidatos.extend(
        [
            Path.cwd(),
            *Path.cwd().parents
        ]
    )

    for candidato in candidatos:
        if (candidato / MARCADOR_REPOSITORIO).is_file():
            return candidato.resolve()

    raise RuntimeError(
        "No se pudo localizar la raíz del repositorio. "
        "Ejecute el notebook desde el repositorio o "
        "defina la variable MNA_REPO_DIR."
    )


REPO_DIR = find_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repositorio localizado correctamente")

Repositorio localizado correctamente


### 7.3 Verificación del entorno experimental

Se registra la configuración básica del entorno en el que se ejecutarán los experimentos, incluyendo versiones de las principales bibliotecas, disponibilidad de CUDA, dispositivo de cómputo y configuración determinista de cuDNN.

Esta información permite identificar cambios de entorno que pudieran afectar la reproducibilidad de los resultados.

In [3]:
# ============================================================
# Dispositivo y versiones del entorno
# ============================================================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    nombre_dispositivo = torch.cuda.get_device_name(0)

    memoria_total_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )
else:
    DEVICE = torch.device("cpu")
    nombre_dispositivo = "CPU"
    memoria_total_gb = np.nan


print("=== ENTORNO EXPERIMENTAL ===")
print("Semilla:", SEED)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
print("Hardware:", nombre_dispositivo)

if torch.cuda.is_available():
    print(
        "Memoria GPU total:",
        f"{memoria_total_gb:.2f} GB"
    )

print(
    "cuDNN deterministic:",
    torch.backends.cudnn.deterministic
)

print(
    "cuDNN benchmark:",
    torch.backends.cudnn.benchmark
)
print("Entrenamiento habilitado:", RUN_TRAINING)

=== ENTORNO EXPERIMENTAL ===
Semilla: 42
Python: 3.12.3
NumPy: 2.5.2
Pandas: 3.0.6
PyTorch: 2.14.1+cu126
Torchvision: 0.29.1+cu126
Scikit-learn: 1.9.1
CUDA disponible: True
Dispositivo: cuda
Hardware: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memoria GPU total: 6.00 GB
cuDNN deterministic: True
cuDNN benchmark: False
Entrenamiento habilitado: False


In [4]:
from Scripts.validar_artefactos_avance5 import (
    validar_avance4
)

tabla_artefactos = validar_avance4(REPO_DIR)

display(tabla_artefactos)

(          modelo                    checkpoint  \
 0  R(2+1)D-18/H5     r2plus1d_18_avance4_h5.pt   
 1      MC3-18/H5          mc3_18_avance4_h5.pt   
 2      MViT-V2-S  mvit_v2_s_avance4_inicial.pt   
 
                                historial  epoch    val_ap  epocas_historial  \
 0     r2plus1d_18_avance4_h5_history.csv      7  0.755428                15   
 1          mc3_18_avance4_h5_history.csv      6  0.713355                15   
 2  mvit_v2_s_avance4_inicial_history.csv      9  0.647266                15   
 
   estado  
 0     OK  
 1     OK  
 2     OK  ,
 {'sha256': 'b85d6b55e8c32420a4df5a0fb9a10e558105beee47f1d4a98ad83d63fcbda21e',
  'n_registros': 974,
  'n_grupos': 63,
  'particiones': {'train': 702, 'test': 136, 'val': 136},
  'estado': 'OK'})

## 8. Recuperación y validación de artefactos del Avance 4

Los ensambles del Avance 5 reutilizan el dataset y modelos obtenidos en el Avance 4. Antes de utilizarlos se verificará que el dataset corresponda exactamente a la versión v2 congelada, que las particiones mantengan su independencia y que los checkpoints e historiales requeridos estén disponibles y sean compatibles con el experimento.

Esta validación se realizará antes de generar nuevos embeddings o probabilidades. Si algún artefacto requerido falta o presenta inconsistencias, la ejecución se detendrá en lugar de continuar con resultados cuya procedencia no pueda verificarse.

La lógica detallada de validación se mantendrá en `Scripts/`, mientras que el notebook mostrará únicamente las decisiones experimentales y los resultados necesarios para comprobar que los artefactos pueden reutilizarse.

### Validación de artefactos heredados

Se ejecuta la validación del dataset y de los artefactos provenientes del Avance 4 antes de utilizarlos en los ensambles. La comprobación incluye la identidad del dataset, la independencia de los grupos experimentales y la disponibilidad y compatibilidad de los checkpoints e historiales requeridos.

La ejecución se detendrá si se detecta alguna inconsistencia.

In [5]:
import importlib

import Scripts.validar_artefactos_avance5 as validar_avance5

importlib.reload(validar_avance5)

tabla_artefactos, resumen_dataset = (
    validar_avance5.validar_avance4(REPO_DIR)
)

display(tabla_artefactos)

print("Dataset v2 validado:", resumen_dataset["n_registros"], "clips elegibles")
print("Grupos experimentales:", resumen_dataset["n_grupos"])
print("Particiones:",resumen_dataset["particiones"])

,modelo,checkpoint,historial,epoch,val_ap,epocas_historial,estado
0,R(2+1)D-18/H5,r2plus1d_18_avance4_h5.pt,r2plus1d_18_avance4_h5_history.csv,7,0.755428,15,OK
1,MC3-18/H5,mc3_18_avance4_h5.pt,mc3_18_avance4_h5_history.csv,6,0.713355,15,OK
2,MViT-V2-S,mvit_v2_s_avance4_inicial.pt,mvit_v2_s_avance4_inicial_history.csv,9,0.647266,15,OK


Dataset v2 validado: 974 clips elegibles
Grupos experimentales: 63
Particiones: {'train': 702, 'test': 136, 'val': 136}


### 8.1 Resultado de la validación de artefactos

Para este avance se reutilizarán los checkpoints e historiales de los modelos finalistas del Avance 4 y se incorporará el checkpoint de MViT-V2-S, cuya arquitectura basada en Transformer aporta diversidad al conjunto de modelos considerado. Se comprobó que los tres checkpoints y sus historiales existen y contienen la información esperada, incluyendo los valores de Average Precision registrados durante la validación.

Todos los experimentos del Avance 5 utilizarán las mismas particiones empleadas en el Avance 4 para mantener comparabilidad y consistencia entre resultados. Los archivos correspondientes al dataset y a sus particiones también fueron localizados y validados antes de continuar con la generación de nuevos artefactos.

## 9. Generación de artefactos para los ensambles

Los modelos de ensamble utilizarán representaciones y predicciones generadas por los modelos RGB validados en la sección anterior.

Para los ensambles homogéneos se extraerán los embeddings producidos por R(2+1)D-18/H5 antes de su clasificador final. Estos vectores se utilizarán como variables de entrada para Random Forest y AdaBoost.

Para los ensambles heterogéneos se obtendrán las probabilidades de ocultamiento generadas por R(2+1)D-18/H5, MC3-18/H5 y MViT-V2-S sobre las mismas observaciones. Estas probabilidades constituirán las entradas para soft voting y blending.

Los artefactos se persistirán junto con `clip_id`, etiqueta, partición y grupo experimental para mantener su trazabilidad. En esta etapa se generarán únicamente datos correspondientes a `train` y `validation`. Los artefactos de `test` se producirán después de congelar las decisiones de modelado que correspondan.

### 9.1 Extracción de representaciones con R(2+1)D-18/H5

Los ensambles Random Forest y AdaBoost utilizarán como variables de entrada la representación interna aprendida por R(2+1)D-18/H5.

El modelo individual convierte cada clip en una representación multidimensional antes de aplicar su clasificador binario final. Para obtener esta representación se recupera primero el checkpoint completo del Avance 4 y posteriormente se sustituye la capa de clasificación por una operación identidad.

Es importante cargar los pesos antes de retirar el clasificador. El checkpoint contiene también los parámetros aprendidos de la capa binaria final y debe restaurarse sobre la misma arquitectura con la que fue entrenado.

Los parámetros del modelo permanecerán congelados durante la extracción. Por lo tanto, esta etapa utiliza R(2+1)D-18/H5 como extractor de características y no realiza entrenamiento adicional.

In [6]:
import torch.nn as nn
from torchvision.models.video import r2plus1d_18


CHECKPOINT_R2PLUS1D_H5 = (
    REPO_DIR / "Notebooks" / "checkpoints" / "r2plus1d_18_avance4_h5.pt"
)

modelo_r2plus1d = r2plus1d_18(weights=None)
dim_clasificador = modelo_r2plus1d.fc.in_features
modelo_r2plus1d.fc = nn.Linear(dim_clasificador, 2)

checkpoint_r2plus1d = torch.load(
    CHECKPOINT_R2PLUS1D_H5,
    map_location=DEVICE,
    weights_only=False
)

modelo_r2plus1d.load_state_dict(checkpoint_r2plus1d["model_state_dict"])

DIM_EMBEDDING_R2PLUS1D = modelo_r2plus1d.fc.in_features
modelo_r2plus1d.fc = nn.Identity()

for parametro in modelo_r2plus1d.parameters():
    parametro.requires_grad = False

modelo_r2plus1d = modelo_r2plus1d.to(DEVICE)
modelo_r2plus1d.eval()

print("Modelo:", "R(2+1)D-18/H5")
print("Dimensión del embedding:", DIM_EMBEDDING_R2PLUS1D)
print("Clasificador para extracción:", modelo_r2plus1d.fc)
print("Parámetros entrenables:", sum(p.numel() for p in modelo_r2plus1d.parameters() if p.requires_grad))

Modelo: R(2+1)D-18/H5
Dimensión del embedding: 512
Clasificador para extracción: Identity()
Parámetros entrenables: 0


Para generar los embeddings se recuperan las mismas particiones de entrenamiento y validación utilizadas en el Avance 4. Solo se consideran los registros marcados como elegibles para el modelado.

El orden de las filas se conserva durante todo el proceso de extracción, ya que posteriormente se utilizará para asociar cada representación con su `clip_id`, grupo experimental, partición y etiqueta.

In [7]:
SPLIT_V2_PATH = REPO_DIR / "Datos" / "Preparacion_Datasets" / "avance4_split_ground_truth_v2.csv"

df_split_v2 = pd.read_csv(SPLIT_V2_PATH)
df_elegibles_v2 = df_split_v2.loc[df_split_v2["elegible_avance4"]].copy()

df_train_v2 = df_elegibles_v2.loc[df_elegibles_v2["split"] == "train"].reset_index(drop=True)
df_val_v2 = df_elegibles_v2.loc[df_elegibles_v2["split"] == "val"].reset_index(drop=True)

print("Train:", len(df_train_v2), "clips")
print("Validation:", len(df_val_v2), "clips")
print("Train únicos:", df_train_v2["clip_id"].nunique())
print("Validation únicos:", df_val_v2["clip_id"].nunique())

Train: 702 clips
Validation: 136 clips
Train únicos: 702
Validation únicos: 136


Para conservar las mismas condiciones de preprocesamiento del Avance 4, los clips se muestrean utilizando 16 frames distribuidos uniformemente y se aplica la transformación oficial de R(2+1)D-18.

Los loaders utilizados para generar embeddings mantienen `shuffle=False`. De esta forma, el orden de las representaciones obtenidas coincide con el orden de las filas de cada partición y puede conservarse la trazabilidad con los metadatos originales.

In [8]:
import os
import cv2

from torch.utils.data import Dataset, DataLoader
from torchvision.models import video as video_models


NUM_FRAMES = 16
BATCH_INFERENCIA = 16


def cargar_frames_rgb(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el clip: {video_path.name}")

    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if n_frames < num_frames:
        cap.release()
        raise RuntimeError(
            f"El clip {video_path.name} contiene {n_frames} frames y requiere al menos {num_frames}."
        )

    indices = np.round(np.linspace(0, n_frames - 1, num_frames)).astype(int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()

        if not ret:
            cap.release()
            raise RuntimeError(f"No se pudo leer el frame {idx} del clip {video_path.name}")

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(torch.from_numpy(frame.copy()).permute(2, 0, 1))

    cap.release()
    return torch.stack(frames, dim=0)


class VideoDatasetRGB(Dataset):
    def __init__(self, dataframe, clips_dir, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.clips_dir = Path(clips_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fila = self.df.iloc[idx]
        video_path = self.clips_dir / fila["nombre_clip"]

        if not video_path.exists():
            raise FileNotFoundError(f"No se encontró el clip requerido: {video_path.name}")

        clip = cargar_frames_rgb(video_path)

        if self.transform is not None:
            clip = self.transform(clip)

        etiqueta = torch.tensor(int(fila["clase_binaria"]), dtype=torch.long)
        return clip, etiqueta


bnext_path = os.environ.get("BNEXT_CLIPS_DIR")

if bnext_path is None:
    raise RuntimeError("La variable de entorno BNEXT_CLIPS_DIR no está configurada.")

CLIPS_DIR = Path(bnext_path)

if not CLIPS_DIR.is_dir():
    raise RuntimeError("El directorio configurado en BNEXT_CLIPS_DIR no está disponible.")

transform_r2plus1d = video_models.R2Plus1D_18_Weights.DEFAULT.transforms()

dataset_embeddings_train = VideoDatasetRGB(df_train_v2, CLIPS_DIR, transform_r2plus1d)
dataset_embeddings_val = VideoDatasetRGB(df_val_v2, CLIPS_DIR, transform_r2plus1d)

loader_embeddings_train = DataLoader(
    dataset_embeddings_train,
    batch_size=BATCH_INFERENCIA,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

loader_embeddings_val = DataLoader(
    dataset_embeddings_val,
    batch_size=BATCH_INFERENCIA,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Batches train:", len(loader_embeddings_train))
print("Batches validation:", len(loader_embeddings_val))
print("Shuffle:", False)

Batches train: 44
Batches validation: 9
Shuffle: False


Antes de generar todos los embeddings se verifica la salida del extractor sobre un batch de entrenamiento. Al sustituir el clasificador final por `Identity()`, cada clip debe producir directamente la representación interna de 512 características aprendida por R(2+1)D-18/H5.

Esta comprobación permite detectar problemas de dimensiones o configuración antes de procesar las particiones completas.

In [9]:
batch_clips, batch_labels = next(iter(loader_embeddings_train))
batch_clips = batch_clips.to(DEVICE, non_blocking=True)

with torch.inference_mode():
    batch_embeddings = modelo_r2plus1d(batch_clips)

print("Forma de entrada:", tuple(batch_clips.shape))
print("Forma de embeddings:", tuple(batch_embeddings.shape))
print("Etiquetas:", tuple(batch_labels.shape))
print("Valores finitos:", bool(torch.isfinite(batch_embeddings).all()))

Forma de entrada: (16, 3, 16, 112, 112)
Forma de embeddings: (16, 512)
Etiquetas: (16,)
Valores finitos: True


La extracción se realiza por batches con el modelo en modo de evaluación y sin cálculo de gradientes. Para cada batch, R(2+1)D-18/H5 transforma los clips en vectores de 512 características. Las representaciones obtenidas se trasladan a CPU y finalmente se concatenan para formar una matriz con una fila por clip.

Las etiquetas se recuperan en el mismo recorrido para comprobar posteriormente que mantienen la correspondencia con los metadatos de la partición original.

In [10]:
def extraer_embeddings(modelo, loader, device):
    embeddings_batches = []
    labels_batches = []

    modelo.eval()

    with torch.inference_mode():
        for clips, labels in loader:
            clips = clips.to(device, non_blocking=True)
            embeddings = modelo(clips)

            embeddings_batches.append(embeddings.cpu())
            labels_batches.append(labels.cpu())

    embeddings = torch.cat(embeddings_batches).numpy()
    labels = torch.cat(labels_batches).numpy()

    return embeddings, labels

Una vez verificada la configuración del extractor, se generan las representaciones para las particiones de entrenamiento y validación. Cada clip queda representado mediante un vector de 512 características producido por R(2+1)D-18/H5.

Después de la extracción se verifica que el número de representaciones y sus etiquetas coincidan exactamente con las observaciones de las particiones originales antes de persistir los artefactos.

In [11]:
import time


inicio = time.perf_counter()

embeddings_train, labels_train = extraer_embeddings(
    modelo_r2plus1d,
    loader_embeddings_train,
    DEVICE
)

embeddings_val, labels_val = extraer_embeddings(
    modelo_r2plus1d,
    loader_embeddings_val,
    DEVICE
)

tiempo_embeddings = time.perf_counter() - inicio

assert embeddings_train.shape == (len(df_train_v2), DIM_EMBEDDING_R2PLUS1D)
assert embeddings_val.shape == (len(df_val_v2), DIM_EMBEDDING_R2PLUS1D)

assert np.array_equal(labels_train, df_train_v2["clase_binaria"].to_numpy())
assert np.array_equal(labels_val, df_val_v2["clase_binaria"].to_numpy())

assert np.isfinite(embeddings_train).all()
assert np.isfinite(embeddings_val).all()

print("Embeddings train:", embeddings_train.shape)
print("Embeddings validation:", embeddings_val.shape)
print("Etiquetas alineadas:", True)
print("Valores finitos:", True)
print("Tiempo de extracción:", round(tiempo_embeddings, 2), "s")

Embeddings train: (702, 512)
Embeddings validation: (136, 512)
Etiquetas alineadas: True
Valores finitos: True
Tiempo de extracción: 273.29 s
